# 7. Iterative Fix Loop (진단→치환→재평가→종료)

## 이번 노트북에서 할 것
- `molecule_editor.propose_fix()`를 반복 호출하는 루프 작성
  (진단 → 치환 → 재평가 → 남은 문제 있으면 반복)
- 종료조건 3가지 구현: ① 문제 다 해결되면 성공 종료 ② 최대 반복 횟수 도달 ③ 순환 감지(이전에 나왔던 분자로 되돌아오면 중단)
- 반복 중 "두 치환이 합쳐져 새 문제가 생기는" 경우도 매 반복 전체 재검사로 자연스럽게 잡히는지 확인

## 간략한 정리 (06까지)
- 도구 계층 완전히 완성됨:
  - `data_prep.py`: Tox21 로드+필터링+SMILES 반환
  - baseline model: ECFP+RandomForest, AUROC 0.821 (ChemBERTa 0.787보다 우수, 채택 근거 있음)
  - `toxicophore_detector.py`: FilterCatalog(PAINS+BRENK) → 문제구조명+원자인덱스
  - `replacement_library.py`: 5개 규칙별 치환 후보(+rationale)
  - `molecule_editor.py`: `find_core_and_target()` + `reassemble_molecule()` + `propose_fix()` — 문제구조 하나를 실제로 치환해서 새 분자 생성, 검증 완료
- 한계로 확인된 것: `propose_fix()`는 한 번에 문제 하나만 고침 (여러 문제 있으면 순차 처리 필요 → 오늘 할 일)
- "치환 조합으로 새 독성 생기는 문제"는 별도 예측 로직 없이, 매 반복마다 전체 재검사(`detect_toxicophores`)로 해결하기로 결정함

## 다음에 해야 할 것 (오늘 루프까지 끝나면)
- 이 반복 루프가 곧 "에이전트"의 뼈대가 됨 — 다음은 여기에 LLM 판단(여러 문제 중 뭘 먼저 고칠지, 여러 후보 중 뭘 고를지)을 얹는 단계
- 평가셋(held-out 분자들)에 이 루프를 돌려서 실제로 몇 %가 개선되는지 측정
- 제안서 초안 작성 시작 (마감 8/7)

In [1]:
!pip install rdkit -q

In [2]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

fatal: destination path 'laidd-2026' already exists and is not an empty directory.
/content/laidd-2026
/content/laidd-2026


In [3]:
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix

data = load_tox21_clean()

# 어제 썼던 테스트 분자 다시 찾기 (nitro_group 포함)
test_mol_smiles = None
for s in data['smiles_train'][:500]:
    result = detect_toxicophores(s)
    if any(r['rule_name'] == 'nitro_group' for r in result):
        test_mol_smiles = s
        break

print("테스트 분자:", test_mol_smiles)
print("초기 문제:", detect_toxicophores(test_mol_smiles))

[06:47:40] WARNING: not removing hydrogen atom without neighbors
[06:47:41] Explicit valence for atom # 8 Al, 6, is greater than permitted
[06:47:41] Explicit valence for atom # 3 Al, 6, is greater than permitted
[06:47:41] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:47:41] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:47:41] Explicit valence for atom # 9 Al, 6, is greater than permitted
[06:47:41] Explicit valence for atom # 5 Al, 6, is greater than permitted
[06:47:42] Explicit valence for atom # 16 Al, 6, is greater than permitted
[06:47:42] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[06:47:42] WARNING: not removing hydrogen atom without neighbors


테스트 분자: O=[N+]([O-])c1ccc(C=NO)o1
초기 문제: [{'rule_name': 'imine_1', 'atom_indices': [7, 8]}, {'rule_name': 'nitro_group', 'atom_indices': [0, 1, 2]}, {'rule_name': 'oxime_1', 'atom_indices': [7, 8, 9]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [1, 2]}]


In [5]:
def canonicalize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None

In [6]:
from src.tools.replacement_library import get_replacement_candidates

def iterative_fix_loop(smiles, max_iterations=10, candidate_idx=0):
    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current, "problems": detect_toxicophores(current)}]
    skipped_rules = []  # 우리가 모르는 규칙이라 건너뛴 기록

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        # 우리 라이브러리가 아는 규칙만 후보로 추림
        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])

        if not known_problems:
            # 아는 규칙이 하나도 없으면 더 이상 진행 불가
            return {"status": "no_known_fix", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        target_rule = known_problems[0]['rule_name']
        fixed = propose_fix(current, target_rule, candidate_idx)

        if fixed is None or not fixed['is_valid']:
            # 알고는 있지만 이번엔 화학적으로 치환 자체가 실패한 경우
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "candidate_used": fixed['candidate_used'],
            "problems": detect_toxicophores(current),
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

In [7]:
result = iterative_fix_loop(test_mol_smiles, max_iterations=10)
print("최종 상태:", result['status'])
print("최종 분자:", result['final_smiles'])
print("\n스텝별 기록:")
for h in result['history']:
    print(h)

최종 상태: no_known_fix
최종 분자: Nc1ccc(C=NO)o1

스텝별 기록:
{'step': 0, 'smiles': 'O=[N+]([O-])c1ccc(C=NO)o1', 'problems': [{'rule_name': 'imine_1', 'atom_indices': [7, 8]}, {'rule_name': 'nitro_group', 'atom_indices': [0, 1, 2]}, {'rule_name': 'oxime_1', 'atom_indices': [7, 8, 9]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [1, 2]}]}
{'step': 1, 'smiles': 'Nc1ccc(C=NO)o1', 'fixed_rule': 'nitro_group', 'candidate_used': 'primary amine', 'problems': [{'rule_name': 'imine_1', 'atom_indices': [5, 6]}, {'rule_name': 'oxime_1', 'atom_indices': [5, 6, 7]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [6, 7]}]}


In [8]:
from collections import Counter

rule_counter = Counter()
for s in data['smiles_train'][:1000]:
    for p in detect_toxicophores(s):
        rule_counter[p['rule_name']] += 1

for rule, count in rule_counter.most_common(15):
    print(f"{rule}: {count}회")

Aliphatic_long_chain: 133회
Oxygen-nitrogen_single_bond: 70회
isolated_alkene: 59회
nitro_group: 44회
alkyl_halide: 44회
aniline: 43회
Sulfonic_acid_2: 36회
imine_1: 31회
Michael_acceptor_1: 30회
aldehyde: 28회
phosphor: 21회
quaternary_nitrogen_1: 17회
beta-keto/anhydride: 17회
quaternary_nitrogen_2: 16회
thiol_2: 12회


In [9]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "michael_acceptor": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "thiourea": {
        "problem_smarts": "NC(=S)N",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "황 원자를 산소로 대체, 유사한 형태를 유지하면서 반응성/대사 우려 감소"},
        ],
    },
    "acyl_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2][c]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [10]:
import importlib
import src.tools.replacement_library
importlib.reload(src.tools.replacement_library)
from src.tools.replacement_library import get_replacement_candidates

print(get_replacement_candidates("alkyl_halide"))
print(get_replacement_candidates("aniline"))

{'problem_smarts': '[Cl,Br,I]', 'candidates': [{'smiles': 'O', 'name': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지'}, {'smiles': 'F', 'name': 'fluorine', 'rationale': '할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사'}]}
{'problem_smarts': '[NH2][c]', 'candidates': [{'smiles': 'C(=O)N', 'name': 'acetamide (acylated amine)', 'rationale': '1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단'}, {'smiles': 'F', 'name': 'fluorine', 'rationale': '반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정'}]}


In [11]:
result2 = iterative_fix_loop(test_mol_smiles, max_iterations=10)
print("최종 상태:", result2['status'])
print("최종 분자:", result2['final_smiles'])
print("건너뛴 규칙들:", result2['skipped_rules'])
print("\n스텝별 기록:")
for h in result2['history']:
    print(h)

최종 상태: no_known_fix
최종 분자: Nc1ccc(C=NO)o1
건너뛴 규칙들: ['imine_1', 'oxime_1', 'Oxygen-nitrogen_single_bond']

스텝별 기록:
{'step': 0, 'smiles': 'O=[N+]([O-])c1ccc(C=NO)o1', 'problems': [{'rule_name': 'imine_1', 'atom_indices': [7, 8]}, {'rule_name': 'nitro_group', 'atom_indices': [0, 1, 2]}, {'rule_name': 'oxime_1', 'atom_indices': [7, 8, 9]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [1, 2]}]}
{'step': 1, 'smiles': 'Nc1ccc(C=NO)o1', 'fixed_rule': 'nitro_group', 'candidate_used': 'primary amine', 'problems': [{'rule_name': 'imine_1', 'atom_indices': [5, 6]}, {'rule_name': 'oxime_1', 'atom_indices': [5, 6, 7]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [6, 7]}]}


In [12]:
# alkyl_halide가 걸리는 분자 찾기
alkyl_test = None
for s in data['smiles_train'][:1000]:
    result = detect_toxicophores(s)
    if any(r['rule_name'] == 'alkyl_halide' for r in result):
        alkyl_test = s
        break

print("alkyl_halide 테스트 분자:", alkyl_test)
if alkyl_test:
    result3 = iterative_fix_loop(alkyl_test, max_iterations=10)
    print("상태:", result3['status'])
    print("최종 분자:", result3['final_smiles'])
    for h in result3['history']:
        print(h)

alkyl_halide 테스트 분자: O=C(O)CCl
상태: success
최종 분자: O=C(O)CO
{'step': 0, 'smiles': 'O=C(O)CCl', 'problems': [{'rule_name': 'alkyl_halide', 'atom_indices': [3, 4]}]}
{'step': 1, 'smiles': 'O=C(O)CO', 'fixed_rule': 'alkyl_halide', 'candidate_used': 'hydroxyl (alcohol)', 'problems': []}


In [13]:
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix
core_test = find_core_and_target("O=C(O)CCl", "alkyl_halide")
print(core_test)

{'core': 'O=C(O)C[*:1]', 'target_removed': 'Cl[*:1]'}


In [14]:
import importlib
import src.tools.replacement_library
importlib.reload(src.tools.replacement_library)
from src.tools.replacement_library import get_replacement_candidates

print(get_replacement_candidates("alkyl_halide"))

{'problem_smarts': '[Cl,Br,I]', 'candidates': [{'smiles': 'O', 'name': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지'}, {'smiles': 'F', 'name': 'fluorine', 'rationale': '할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사'}]}


In [15]:
import src.tools.molecule_editor
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target

core_test3 = find_core_and_target("O=C(O)CCl", "alkyl_halide")
print(core_test3)

{'core': 'O=C(O)C[*:1]', 'target_removed': 'Cl[*:1]'}


In [16]:
import src.tools.molecule_editor
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target, propose_fix

core_test3 = find_core_and_target("O=C(O)CCl", "alkyl_halide")
print(core_test3)

{'core': 'O=C(O)C[*:1]', 'target_removed': 'Cl[*:1]'}


def find_core_and_target(smiles: str, rule_name: str):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()  # 패턴이 정의하는 원자 개수

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)

    best_match = None
    for core, chain in fragments:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            part_mol = Chem.MolFromSmiles(part.replace('[*:1]', '[H]'))
            if part_mol is None:
                continue
            if not part_mol.HasSubstructMatch(problem_pattern):
                continue
            # 조각의 원자 수(수소 제외)가 패턴 크기와 정확히 일치하는 것만 인정
            frag_heavy_atoms = part_mol.GetNumHeavyAtoms()
            if frag_heavy_atoms == pattern_size:
                return {"core": parts[1 - i], "target_removed": part}
            # 정확히 일치하진 않지만 후보로 남겨둠 (더 나은 게 없으면 이거라도 씀)
            if best_match is None:
                best_match = {"core": parts[1 - i], "target_removed": part}
    return best_match

In [17]:
!cat src/tools/molecule_editor.py

from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환. 못 찾으면 None."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)

    best_match = None
    for core, chain in fragments:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            part_mol = Chem.MolFromSmiles(part.replace('[*:1]', '[H]'))
            if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
 

In [18]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환. 못 찾으면 None."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)

    best_match = None
    for core, chain in fragments:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            part_mol = Chem.MolFromSmiles(part.replace('[*:1]', '[H]'))
            if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
                continue
            frag_heavy_atoms = part_mol.GetNumHeavyAtoms()
            if frag_heavy_atoms == pattern_size:
                return {"core": parts[1 - i], "target_removed": part}
            if best_match is None:
                best_match = {"core": parts[1 - i], "target_removed": part}
    return best_match


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    """core의 [*:1] 자리에 replacement_library의 candidate를 붙여 새 분자를 완성."""
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    """전체 파이프라인: 문제구조 위치 찾기 -> 치환 후보로 재조립까지 한번에 실행."""
    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    """SMILES를 canonical(정규) 형태로 변환. 파싱 실패 시 None."""
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0):
    """진단->치환->재평가를 반복. 성공/실패/순환/미지의 규칙 등으로 종료."""
    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []

    for step in range(1, max_iterations + 1):
        # 순환참조 방지를 위해 여기서 import (같은 파일 내 함수는 아래에서 직접 씀)
        from src.tools.toxicophore_detector import detect_toxicophores
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        target_rule = known_problems[0]['rule_name']
        fixed = propose_fix(current, target_rule, candidate_idx)

        if fixed is None or not fixed['is_valid']:
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "candidate_used": fixed['candidate_used'],
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

Overwriting src/tools/molecule_editor.py


In [19]:
import importlib
import src.tools.molecule_editor
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target, propose_fix, iterative_fix_loop

# 1) 조각 분리가 이번엔 제대로 되는지
core_test4 = find_core_and_target("O=C(O)CCl", "alkyl_halide")
print("core_test4:", core_test4)

# 2) 전체 치환 결과
fix_test = propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0)
print("fix_test:", fix_test)

core_test4: {'core': 'O=C(O)C[*:1]', 'target_removed': 'Cl[*:1]'}
fix_test: {'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}


In [20]:
result5 = iterative_fix_loop(test_mol_smiles, max_iterations=10)
print("최종 상태:", result5['status'])
print("최종 분자:", result5['final_smiles'])
print("건너뛴 규칙:", result5['skipped_rules'])
for h in result5['history']:
    print(h)

최종 상태: no_known_fix
최종 분자: Nc1ccc(C=NO)o1
건너뛴 규칙: ['imine_1', 'oxime_1', 'Oxygen-nitrogen_single_bond']
{'step': 0, 'smiles': 'O=[N+]([O-])c1ccc(C=NO)o1', 'problems': [{'rule_name': 'imine_1', 'atom_indices': [7, 8]}, {'rule_name': 'nitro_group', 'atom_indices': [0, 1, 2]}, {'rule_name': 'oxime_1', 'atom_indices': [7, 8, 9]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [1, 2]}]}
{'step': 1, 'smiles': 'Nc1ccc(C=NO)o1', 'fixed_rule': 'nitro_group', 'candidate_used': 'primary amine', 'problems': [{'rule_name': 'imine_1', 'atom_indices': [5, 6]}, {'rule_name': 'oxime_1', 'atom_indices': [5, 6, 7]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [6, 7]}]}


In [21]:
result6 = iterative_fix_loop("O=C(O)CCl", max_iterations=10)
print("최종 상태:", result6['status'])
print("최종 분자:", result6['final_smiles'])
for h in result6['history']:
    print(h)

최종 상태: success
최종 분자: O=C(O)CO
{'step': 0, 'smiles': 'O=C(O)CCl', 'problems': [{'rule_name': 'alkyl_halide', 'atom_indices': [3, 4]}]}
{'step': 1, 'smiles': 'O=C(O)CO', 'fixed_rule': 'alkyl_halide', 'candidate_used': 'hydroxyl (alcohol)', 'problems': []}


In [22]:
!git add src/tools/molecule_editor.py src/tools/replacement_library.py
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/molecule_editor.py
	modified:   src/tools/replacement_library.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	laidd-2026/



In [23]:
!git commit -m "Fix find_core_and_target atom-size matching; fix alkyl_halide SMARTS; add iterative_fix_loop with cycle/unknown-rule detection"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main e607239] Fix find_core_and_target atom-size matching; fix alkyl_halide SMARTS; add iterative_fix_loop with cycle/unknown-rule detection
 2 files changed, 84 insertions(+), 2 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 2.74 KiB | 2.74 MiB/s, done.
Total 6 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Dec32th/laidd-2026.git
   9fec1d7..e607239  main -> main
